In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import os
import json
import random
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import httpx
import numpy as np
from tqdm import tqdm


# -----------------------
# Config
# -----------------------
VDB_URL = os.getenv("VDB_SERVICE_URL", "http://localhost:8003")
COLLECTION = os.getenv("VDB_COLLECTION", "finetuning")

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:8000/v1")
LLM_API_KEY = os.getenv("LLM_API_KEY", "EMPTY")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")

EMBEDDER_NAME = os.getenv("EMBEDDER_NAME", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

OUT_DIR = os.getenv("OUT_DIR", "dataset_out")
SEED = int(os.getenv("SEED", "42"))

# Build params (good defaults for papers)
NUM_SEED_CHUNKS = int(os.getenv("NUM_SEED_CHUNKS", "1200"))   # sample chunks to generate queries
QUERIES_PER_CHUNK = int(os.getenv("QUERIES_PER_CHUNK", "2"))  # 2-3 for papers
MIN_CONF = float(os.getenv("MIN_CONF", "0.55"))

HARD_TOP_K = int(os.getenv("HARD_TOP_K", "120"))
NUM_HARD_NEG = int(os.getenv("NUM_HARD_NEG", "8"))
NUM_SIMPLE_NEG = int(os.getenv("NUM_SIMPLE_NEG", "4"))

MAX_DOC_CHARS = int(os.getenv("MAX_DOC_CHARS", "1800"))


# -----------------------
# Helpers
# -----------------------
def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def safe_text(s: str, max_chars: int = 1800) -> str:
    return s.strip()[:max_chars]

def jdump(obj) -> str:
    return json.dumps(obj, ensure_ascii=False)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)


# -----------------------
# Embedder (for mining)
# -----------------------
def build_embedder():
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(EMBEDDER_NAME)

    def encode(text: str) -> List[float]:
        v = model.encode(text, normalize_embeddings=True)
        return v.tolist()

    return encode


# -----------------------
# VDB client (adapt to your API)
# -----------------------
class VDBClient:
    def __init__(self, base_url: str, collection: str):
        self.base_url = base_url.rstrip("/")
        self.collection = collection

    def sample(self, n: int) -> List[Dict[str, Any]]:
        """
        Expect: POST /sample {collection, n} -> {items:[{id,text,meta,...}]}
        If you don't have /sample, replace with your own endpoint.
        """
        with httpx.Client(timeout=120.0) as client:
            r = client.post(f"{self.base_url}/sample", json={"collection": self.collection, "n": n})
            r.raise_for_status()
            return r.json()["items"]

    def get(self, ids: List[str]) -> List[Dict[str, Any]]:
        with httpx.Client(timeout=120.0) as client:
            r = client.post(f"{self.base_url}/get", json={"collection": self.collection, "ids": ids})
            r.raise_for_status()
            return r.json()["items"]

    def search(self, vector: List[float], top_k: int) -> List[Dict[str, Any]]:
        """
        Expect: POST /query {collection, vector, top_k} -> {items:[{id,text,score,meta}]}
        """
        with httpx.Client(timeout=120.0) as client:
            r = client.post(
                f"{self.base_url}/query",
                json={"collection": self.collection, "vector": vector, "top_k": top_k},
            )
            r.raise_for_status()
            return r.json()["items"]

    def random_ids(self, n: int) -> List[str]:
        """
        Expect: POST /random_ids {collection, n} -> {ids:[...]}
        """
        with httpx.Client(timeout=120.0) as client:
            r = client.post(f"{self.base_url}/random_ids", json={"collection": self.collection, "n": n})
            r.raise_for_status()
            return r.json()["ids"]


# -----------------------
# LLM client (OpenAI compatible)
# -----------------------
class LLMClient:
    def __init__(self, base_url: str, api_key: str, model: str):
        self.base_url = base_url.rstrip("/")
        self.headers = {"Authorization": f"Bearer {api_key}"}
        self.model = model

    def chat(self, messages: List[Dict[str, str]], temperature: float = 0.2) -> str:
        with httpx.Client(timeout=120.0) as client:
            r = client.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json={"model": self.model, "messages": messages, "temperature": temperature},
            )
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]

    def generate_queries(self, chunk_text: str, n: int) -> List[Dict[str, Any]]:
        # JSON-only response
        sys = "You create high-quality query/positive pairs for embedding training on scientific papers."
        user = f"""
Generate {n} realistic user queries that are answered by the passage.
For each query, include an answer_quote which is a VERBATIM sentence from the passage that supports the query.
Return strict JSON array of objects:
- query (string)
- answer_quote (string)
- confidence (0..1)

Passage:
\"\"\"{safe_text(chunk_text, MAX_DOC_CHARS)}\"\"\"
"""
        out = self.chat([{"role": "system", "content": sys}, {"role": "user", "content": user}], temperature=0.3)

        # Extract JSON array
        m = re.search(r"\[.*\]", out, flags=re.S)
        if not m:
            return []
        arr = json.loads(m.group(0))
        cleaned = []
        for obj in arr:
            q = normalize_ws(str(obj.get("query", "")))
            quote = str(obj.get("answer_quote", "")).strip()
            conf = float(obj.get("confidence", 0.0))
            if not q or not quote:
                continue
            if quote not in chunk_text:
                continue
            cleaned.append({"query": q, "answer_quote": quote, "confidence": conf})
        return cleaned

    def judge_negative(self, query: str, passage: str) -> bool:
        """
        Return True if passage is safe negative (label 0).
        """
        sys = "You are a strict relevance judge for retrieval."
        user = f"""
Does the passage answer the query?
Return JSON: {{"label":0|1|2}}
2=answers, 1=partially related, 0=does not answer.
Be strict: only 2 if the passage explicitly answers.

Query: {query}

Passage:
\"\"\"{safe_text(passage, MAX_DOC_CHARS)}\"\"\"
"""
        out = self.chat([{"role": "system", "content": sys}, {"role": "user", "content": user}], temperature=0.0)
        m = re.search(r"\{.*\}", out, flags=re.S)
        if not m:
            return False
        obj = json.loads(m.group(0))
        return int(obj.get("label", 0)) == 0


# -----------------------
# Build dataset
# -----------------------
def write_jsonl(path: str, rows: List[Dict[str, Any]]):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(jdump(r) + "\n")


def main():
    set_seed(SEED)

    vdb = VDBClient(VDB_URL, COLLECTION)
    llm = LLMClient(LLM_BASE_URL, LLM_API_KEY, LLM_MODEL)
    encode = build_embedder()

    # 1) sample chunks
    seed_chunks = vdb.sample(NUM_SEED_CHUNKS)

    # 2) generate positives (queries)
    positives = []
    seen_q = set()

    for item in tqdm(seed_chunks, desc="LLM generate queries"):
        text = item.get("text", "") or ""
        meta = item.get("meta") or {}
        doc_id = meta.get("doc_id") or meta.get("paper_id")
        if not text.strip() or not doc_id:
            continue

        gens = llm.generate_queries(text, QUERIES_PER_CHUNK)
        for g in gens:
            if g["confidence"] < MIN_CONF:
                continue
            q = g["query"]
            if q in seen_q:
                continue
            seen_q.add(q)
            positives.append({
                "query": q,
                "pos_id": str(item.get("id")),
                "pos_doc_id": str(doc_id),
            })

    # 3) mine negatives and export triplets
    triplets = []
    grouped = []

    for p in tqdm(positives, desc="Mining negatives"):
        q = p["query"]
        pos_id = p["pos_id"]
        pos_doc_id = p["pos_doc_id"]

        # Fetch positive text
        pos_item = vdb.get([pos_id])[0]
        pos_text = pos_item.get("text", "") or ""
        if not pos_text.strip():
            continue

        # Hard negatives candidates from vector search
        qvec = encode(q)
        hits = vdb.search(qvec, top_k=HARD_TOP_K)

        hard_negs = []
        for h in hits:
            if len(hard_negs) >= NUM_HARD_NEG:
                break
            hid = str(h.get("id"))
            if hid == pos_id:
                continue
            hmeta = h.get("meta") or {}
            hdoc = hmeta.get("doc_id") or hmeta.get("paper_id")
            if not hdoc:
                continue
            if str(hdoc) == str(pos_doc_id):
                continue  # crucial: inter-papers only
            htext = h.get("text", "") or ""
            if not htext.strip():
                continue
            # optional: judge to avoid false negatives
            if llm.judge_negative(q, htext):
                hard_negs.append({"id": hid, "text": htext})

        # Simple negatives: random ids until we get enough from other doc_id
        simple_negs = []
        attempts = 0
        while len(simple_negs) < NUM_SIMPLE_NEG and attempts < 10:
            attempts += 1
            ids = vdb.random_ids(NUM_SIMPLE_NEG * 5)
            items = vdb.get(ids)
            for it in items:
                if len(simple_negs) >= NUM_SIMPLE_NEG:
                    break
                mid = (it.get("meta") or {})
                did = mid.get("doc_id") or mid.get("paper_id")
                if not did or str(did) == str(pos_doc_id):
                    continue
                txt = it.get("text", "") or ""
                if txt.strip():
                    simple_negs.append({"id": str(it.get("id")), "text": txt})

        negatives = hard_negs + simple_negs
        if not negatives:
            continue

        # Triplets (explode)
        for neg in negatives:
            triplets.append({
                "query": q,
                "positive": pos_text,
                "negative": neg["text"],
                "meta": {
                    "pos_id": pos_id,
                    "pos_doc_id": pos_doc_id,
                    "neg_id": neg["id"],
                    "neg_type": "hard" if neg in hard_negs else "simple",
                }
            })

        grouped.append({
            "query": q,
            "positive": [pos_text],
            "negative": [n["text"] for n in negatives],
            "meta": {"pos_id": pos_id, "pos_doc_id": pos_doc_id},
        })

    write_jsonl(os.path.join(OUT_DIR, "train_triplets.jsonl"), triplets)
    write_jsonl(os.path.join(OUT_DIR, "eval_grouped.jsonl"), grouped)
    print(f"✅ Wrote {len(triplets)} triplets -> {OUT_DIR}/train_triplets.jsonl")
    print(f"✅ Wrote {len(grouped)} queries -> {OUT_DIR}/eval_grouped.jsonl")


if __name__ == "__main__":
    main()
